In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns

: 

In [ ]:
# ==========================================
# 1. CARGA Y PREPARACIÓN EN SERIE TEMPORAL
# ==========================================
ruta_archivo = "/content/historial_cambios (2).csv"

try:
    df = pd.read_csv(ruta_archivo)
except FileNotFoundError:
    print(f"Error: No se encontró el archivo.")
    exit()

In [ ]:
# SOLUCIÓN AL FORMATO: Forzar el parseo correcto de tu formato "23/2/2026, 2:34:58 p.m"
if df['fecha_modificacion'].dtype == 'object':
    df['fecha_modificacion'] = df['fecha_modificacion'].str.replace('.', '', regex=False).str.upper()

df['fecha_modificacion'] = pd.to_datetime(df['fecha_modificacion'], format='%d/%m/%Y, %I:%M:%S %p', errors='coerce')

# Si algunas filas fallan por otro formato, usamos mixed como respaldo
en_blanco = df['fecha_modificacion'].isna()
if en_blanco.any():
    df.loc[en_blanco, 'fecha_modificacion'] = pd.to_datetime(df.loc[en_blanco, 'fecha_modificacion'], format='mixed', errors='coerce')

df = df.dropna(subset=['fecha_modificacion'])

# Extraer solo la fecha (sin hora) para agrupar por día
df['fecha'] = df['fecha_modificacion'].dt.date

# Crear la serie temporal: Contar cuántas tareas se hacen por día
serie_temporal = df.groupby('fecha').size().reset_index(name='total_tareas')

# Asegurar que la fecha sea tipo datetime antes de ordenar
serie_temporal['fecha'] = pd.to_datetime(serie_temporal['fecha'])
serie_temporal = serie_temporal.sort_values('fecha').reset_index(drop=True)

In [ ]:
# ==========================================
# 2. CREACIÓN DE CARACTERÍSTICAS (Feature Engineering)
# ==========================================
serie_temporal['dia_semana'] = serie_temporal['fecha'].dt.dayofweek
serie_temporal['dia_mes'] = serie_temporal['fecha'].dt.day

# "Lag Features" + "Rolling Window" (Media móvil para evitar el estancamiento en cero)
serie_temporal['tareas_ayer'] = serie_temporal['total_tareas'].shift(1)
serie_temporal['media_movil_3d'] = serie_temporal['total_tareas'].shift(1).rolling(window=3).mean()

# Eliminamos filas con NaN resultantes del desfase (shift y rolling)
serie_temporal = serie_temporal.dropna().reset_index(drop=True)

# Variables predictoras (X) y variable a predecir (y)
X = serie_temporal[['dia_semana', 'dia_mes', 'tareas_ayer', 'media_movil_3d']]
y = serie_temporal['total_tareas']

# Función nativa para calcular el wMAPE (Weighted MAPE)
def calcular_wmape(y_real, y_pred):
    sum_real = np.sum(y_real)
    if sum_real == 0:
        return 0.0
    return np.sum(np.abs(y_real - y_pred)) / sum_real


In [ ]:
# ==========================================
# 2.5 EVALUACIÓN ESTRICTA (División Temporal)
# ==========================================
dias_test = 7

if len(serie_temporal) > dias_test * 2:
    X_train, X_test = X.iloc[:-dias_test], X.iloc[-dias_test:]
    y_train, y_test = y.iloc[:-dias_test], y.iloc[-dias_test:]
    
    # Evaluar con un modelo interno entrenado solo con el pasado
    modelo_evaluacion = LinearRegression()
    modelo_evaluacion.fit(X_train, y_train)
    y_pred_test = np.clip(modelo_evaluacion.predict(X_test), 0, None)
    
    # Cálculo de métricas estrictas y realistas
    mae = mean_absolute_error(y_test, y_pred_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
    wmape = calcular_wmape(y_test, y_pred_test)
    precision_ajustada = max(0, 100 - (wmape * 100))
    
    # Guardar predicciones de test en el dataframe para la gráfica
    serie_temporal.loc[serie_temporal.index[-dias_test:], 'prediccion_test'] = y_pred_test
else:
    print("⚠️ Datos insuficientes para realizar una validación temporal estricta.")
    mae, rmse, wmape, precision_ajustada = None, None, None, None


In [ ]:
# ==========================================
# 3. ENTRENAMIENTO DEL MODELO DE PRONÓSTICO (Producción)
# ==========================================
modelo_futuro = LinearRegression()
modelo_futuro.fit(X, y)

# ==========================================
# 4. PREDICCIÓN DE LOS PRÓXIMOS 7 DÍAS (Proyección Dinámica)
# ==========================================
ultima_fecha = serie_temporal['fecha'].max()

# Necesitamos mantener un registro de los últimos valores reales para calcular las variables dinámicamente
historial_valores = list(serie_temporal['total_tareas'].iloc[-3:]) 

futuras_fechas = []
predicciones_futuras = []

for i in range(1, 8):
    nueva_fecha = ultima_fecha + pd.Timedelta(days=i)
    n_dia_semana = nueva_fecha.dayofweek
    n_dia_mes = nueva_fecha.day
    
    # Extraer las características basadas en la ventana móvil simulada
    lag_1 = historial_valores[-1]
    movil_3d = np.mean(historial_valores[-3:])
    
    entrada_prediccion = pd.DataFrame([{
        'dia_semana': n_dia_semana,
        'dia_mes': n_dia_mes,
        'tareas_ayer': lag_1,
        'media_movil_3d': movil_3d
    }])
    
    # Predecir de forma recursiva
    prediccion = max(0, int(modelo_futuro.predict(entrada_prediccion)[0]))
    
    futuras_fechas.append(nueva_fecha)
    predicciones_futuras.append(prediccion)
    
    # Actualizar el historial para la predicción del día siguiente
    historial_valores.append(prediccion)

df_futuro = pd.DataFrame({'fecha': futuras_fechas, 'pronostico': predicciones_futuras})

# Imprimir reporte de calificación corregido
print("\n📊 REPORTE DE CALIFICACIÓN DEL MODELO (Últimos 7 días históricos):")
print("-" * 65)
if mae is not None:
    print(f"• Error Absoluto Medio (MAE):         {mae:.2f} tareas")
    print(f"• Raíz del Error Cuadrático (RMSE):   {rmse:.2f} tareas")
    print(f"• Error Porcentual Ponderado (wMAPE): {wmape * 100:.2f}%")
    print(f"• Precisión General Ajustada:         {precision_ajustada:.2f}%")
else:
    print("No se pudo calcular por falta de histórico suficiente.")
print("-" * 65)

print("\n📈 PRONÓSTICO DE CARGA DE TRABAJO (PRÓXIMOS 7 DÍAS REALES):")
for f, p in zip(futuras_fechas, predicciones_futuras):
    print(f"Fecha: {f.strftime('%Y-%m-%d')} -> Tareas estimadas: {p}")

In [ ]:
# ==========================================
# 5. GRÁFICA DE TENDENCIA, VALIDACIÓN Y PRONÓSTICO
# ==========================================
plt.figure(figsize=(14, 6))
sns.set_theme(style="whitegrid")

# 1. Datos históricos reales
plt.plot(serie_temporal['fecha'], serie_temporal['total_tareas'], label='Actividad Real Pasada', color='#2471a3', marker='o', linewidth=2)

# 2. El comportamiento del modelo en el set de pruebas (Test)
if mae is not None:
    df_test_plot = serie_temporal.dropna(subset=['prediccion_test'])
    plt.plot(df_test_plot['fecha'], df_test_plot['prediccion_test'], label='Simulación / Evaluación del Modelo', color='#f39c12', linestyle=':', marker='X', linewidth=2)

# 3. Línea del pronóstico futuro real
plt.plot(df_futuro['fecha'], df_futuro['pronostico'], label='Pronóstico Futuro Real (7 días)', color='#e74c3c', linestyle='--', marker='s', linewidth=2)

# Configuración visual optimizada
plt.title('Calibración y Proyección de Carga de Trabajo (Corregido con Ventana Móvil)', fontsize=14, weight='bold')
plt.xlabel('Fecha', fontsize=12)
plt.ylabel('Cantidad de Modificaciones / Tareas', fontsize=12)
plt.gcf().autofmt_xdate() 
plt.legend(fontsize=11, loc='upper left')
plt.tight_layout()
plt.show()